In [ ]:
%load_ext autoreload
%autoreload 2
# Reloads lago_agent_sdk automatically whenever its source changes, so fixes
# take effect on the next cell run — no kernel restart needed. Only helps for
# edits made AFTER this cell has run once in the current kernel.

import json
import os
import sys

sys.path.insert(0, "../src")  # run this notebook from examples/, or adjust to your install


def _load_dotenv(path: str) -> None:
    """No extra dependency — just KEY=VALUE lines, same as python-dotenv's basics."""
    if not os.path.exists(path):
        return
    for line in open(path):
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


_load_dotenv(os.path.join(os.getcwd(), ".env"))

from lago_agent_sdk import LagoSDK  # noqa: E402
from lago_agent_sdk.config import LagoConfig  # noqa: E402
from lago_agent_sdk.gateway.databricks import DatabricksSource  # noqa: E402

_REQUIRED = ["DATABRICKS_HOST", "DATABRICKS_TOKEN", "LAGO_API_KEY"]
_missing = [name for name in _REQUIRED if not os.environ.get(name)]
if _missing:
    raise SystemExit(
        f"Missing required environment variable(s): {', '.join(_missing)}.\n"
        "Set them before starting the kernel, or put them in examples/.env — see .env.example."
    )

DBX_HOST = os.environ["DATABRICKS_HOST"].rstrip("/")
DBX_TOKEN = os.environ["DATABRICKS_TOKEN"]
# Backfill only — see Part 1 for the scope it needs.
DBX_WAREHOUSE_ID = os.environ.get("DATABRICKS_WAREHOUSE_ID", "")
# Unity Catalog connections holding your own vendor keys (BYOK). Only needed for
# the provider you actually call in Part 2.
SVC_ANTHROPIC = os.environ.get("DATABRICKS_PROVIDER_SERVICE_ANTHROPIC", "")
SVC_OPENAI = os.environ.get("DATABRICKS_PROVIDER_SERVICE_OPENAI", "")

LAGO_API_KEY = os.environ["LAGO_API_KEY"]
LAGO_API_URL = os.environ.get("LAGO_API_URL", "https://api.getlago.com/api/v1")
LAGO_SUBSCRIPTION_ID = os.environ.get("LAGO_SUBSCRIPTION_ID", "databricks_gateway_demo_sub")
LAGO_VERIFY_SSL = os.environ.get("LAGO_VERIFY_SSL", "true").lower() != "false"

sdk = LagoSDK(
    api_key=LAGO_API_KEY,
    api_url=LAGO_API_URL,
    default_subscription_id=LAGO_SUBSCRIPTION_ID,
    config=LagoConfig(
        api_key=LAGO_API_KEY,
        api_url=LAGO_API_URL,
        pricing_mode="price",
        verify_ssl=LAGO_VERIFY_SSL,
    ),
)
# Blocks until OpenRouter's table is fetched, closing the cold-start race for the
# very first live call. That table is what prices the BYOK paths in Part 2 —
# verified exact against Databricks' own metered spend on 39 of 39 real calls.
# Databricks-HOSTED models are deliberately NOT priceable from it: they bill in
# DBUs against Databricks' own rate card, so they fall back to token events by
# design rather than being priced at some other vendor's rate for the same
# open-weight model.
sdk.warm_pricing()
print("SDK ready — billing to", LAGO_SUBSCRIPTION_ID)
print("backfill enabled:", bool(DBX_WAREHOUSE_ID))


## Part 1 — Backfill historic usage from Databricks

Databricks has no REST logs API. Usage lands in Unity Catalog system tables read
over SQL, and cost lives in a *different* table from the token counts:

| | table | unit |
|---|---|---|
| tokens, per request, with your attribution tags | `system.ai_gateway.usage` | counts |
| cost for BYOK providers, already attributed | `system.ai_gateway.external_model_spend` | **USD** |

`DatabricksSource` reads both and reconciles them, so the whole backfill is a
window plus one call. Doing it by hand is about a hundred lines with three
money-losing traps in them: the Statement Execution API returns only **chunk 0**
inline, so a wide window silently truncates; a BYOK call appears in **both** tables
and billing both charges twice; and `transaction_id` is unique account-wide, so an
unscoped row id blocks that row from ever reaching a second subscription.

The billing rule is the same one the Cloudflare connector follows: **the gateway is
the metering authority**, so a BYOK row bills from Databricks' own `usage_quantity`
via `emit(usd_cost=...)` rather than from a price we compute ourselves.
Databricks-hosted models have no per-request USD anywhere in Databricks' system
tables, so they bill as token events.

`request_tags` is a first-class aggregation dimension on the spend table, so tagging
`lago_subscription` on the call means cost arrives **already split per
subscription** — no apportioning by token share.

One caveat that is easy to miss: a SQL warehouse is a real cost centre. Measured on
the test workspace behind this notebook, warehouse queries cost roughly **1,500x**
the model-serving usage they were reporting on. Read one wide window per run; never
poll in a tight loop.


In [ ]:
# Needs a PAT carrying the `sql` scope plus a SQL warehouse — the live calls in
# Part 2 need neither. A token without `sql` fails every warehouse route with
# 403 "does not have required scopes: sql", including the Thrift path the
# databricks-sql-connector uses, so switching client libraries does not help.
if not DBX_WAREHOUSE_ID:
    raise SystemExit("DATABRICKS_WAREHOUSE_ID is unset — backfill needs a SQL warehouse.")

source = DatabricksSource(host=DBX_HOST, token=DBX_TOKEN, warehouse_id=DBX_WAREHOUSE_ID)
# ...or DatabricksSource.from_env(), which reads the same three variables.

WINDOW = "7 days"  # or a datetime, for an exact lower bound

rows = list(source.read_usage(WINDOW))
byok = [r for r in rows if r.is_byok]
hosted = [r for r in rows if not r.is_byok]

print(f"{len(rows)} billable rows in the last {WINDOW}")
print(f"  {len(byok):>3} BYOK      ${sum(r.usd_cost for r in byok):.6f} metered by Databricks")
print(f"  {len(hosted):>3} hosted    token counts only — no per-request USD exists")
for r in rows[:5]:
    print(f"    {r.usage.provider:<10} {r.usage.model:<24} {r.subscription or '(untagged)'}")


In [ ]:
# One call: resolve each row's subscription, pick cost-vs-tokens per row, and emit.
# unified=True bills everything to one subscription, ignoring per-call tags — right
# when this gateway's traffic all belongs to one customer. Set it False to respect
# real per-call attribution and fall back to the default only for untagged rows.
# `rows` from the cell above is passed straight in, so the window is read ONCE.
# Handing `source` + WINDOW here instead would re-run both warehouse queries — and a
# warehouse costs ~1,500x the model-serving usage it reports on, so that doubles the
# expensive half of this notebook. It would also let rows land between the two reads,
# making the summary printed above disagree with what was billed.
counts = sdk.backfill_databricks(
    rows,
    default_subscription=LAGO_SUBSCRIPTION_ID,
    unified=True,
)
assert sdk.flush(timeout=30.0), "queue did not flush in time"
print(counts)

# Re-run this cell: every event id is derived from the source row and scoped by
# subscription, so Lago rejects the duplicates instead of billing the window twice.


### Compare it against Databricks

Below is what Databricks' own tables say for this window, next to what was sent to Lago.

They are equal **by construction**, not by luck — a BYOK row bills `usage_quantity`
verbatim via `emit(usd_cost=...)` with no price lookup on our side, and a hosted row
bills the table's own token counts. So read this as a visible restatement, not as an
independent audit.

What makes it *checkable* is the last block: every event carries the grouping key of the
Databricks surface it came from — `endpoint_name` for hosted, `bucket` (the hour) for
BYOK. Group Lago by `endpoint_name` and you get the table below, row for row, next to


In [ ]:
# Everything here comes from the `rows` already read above — no extra query, so this
# cell costs nothing. (A SQL warehouse is expensive relative to the traffic it reports
# on: measured at ~1,500x the model-serving usage in this workspace.)
from collections import defaultdict

byok_usd = sum(r.usd_cost for r in byok)
hosted_in = sum(r.usage.input for r in hosted)
hosted_out = sum(r.usage.output for r in hosted)

n_cost = counts["cost"]
n_in = sum(1 for r in hosted if r.usage.input)
n_out = sum(1 for r in hosted if r.usage.output)

print("Databricks says                             ->  sent to Lago")
print(f"BYOK    external_model_spend   ${byok_usd:>8.6f}  ->  llm_cost          "
      f"${byok_usd:>8.6f}   {n_cost} events")
print(f"hosted  ai_gateway.usage       in {hosted_in:>9,}  ->  llm_input_tokens   "
      f"{hosted_in:>10,}   {n_in} events")
print(f"{'':31}out {hosted_out:>8,}  ->  llm_output_tokens  "
      f"{hosted_out:>10,}   {n_out} events")

# Keyed by endpoint_name — the same column the AI Gateway usage page groups by, and the
# dimension now on every hosted event.
per_endpoint = defaultdict(lambda: [0, 0])
for r in hosted:
    key = r.usage.extras.get("endpoint_name") or r.usage.model
    per_endpoint[key][0] += r.usage.input
    per_endpoint[key][1] += r.usage.output

print("\nper endpoint — read against the AI Gateway usage page:")
for endpoint, (tin, tout) in sorted(per_endpoint.items(), key=lambda kv: -sum(kv[1])):
    print(f"  {endpoint:<40} in {tin:>8,}   out {tout:>8,}")


## Part 2 — Live calls through the gateway

Each provider is reachable **only** through its own native surface — there is no
single OpenAI-compatible front door the way Cloudflare offers `/compat`. So the
same `openai.OpenAI` client means two different things depending on `base_url`:
`/ai-gateway/mlflow/v1` is a Databricks-hosted model billed in DBUs, while
`/ai-gateway/openai/v1` is your own OpenAI account.

`Databricks-Ai-Gateway-Request-Tags` carries the Lago attribution and is what
makes cost arrive pre-split per subscription in Part 1.


In [ ]:
PROMPT = "Tell me about getLago, the billing company - give as many details as you can find"
TAGS = json.dumps({"lago_subscription": LAGO_SUBSCRIPTION_ID, "team": "lago-demo"})


In [ ]:
# Databricks-HOSTED foundation model, via the unified mlflow surface.
# Prices in DBUs against Databricks' own rate card, which OpenRouter does not
# carry — so this bills as token events, deliberately, rather than being matched
# to some other vendor's price for the same open-weight model.
from openai import OpenAI

client = sdk.wrap(OpenAI(
    api_key=DBX_TOKEN,
    base_url=f"{DBX_HOST}/ai-gateway/mlflow/v1",
    default_headers={"Databricks-Ai-Gateway-Request-Tags": TAGS},
))
resp = client.chat.completions.create(
    model="system.ai.llama-4-maverick",
    messages=[{"role": "user", "content": PROMPT}],
    max_tokens=400,
)
text = resp.choices[0].message.content
print("resolved model:", resp.model, "| usage:", resp.usage)


In [ ]:
print(text)


In [ ]:
# Anthropic BYOK, via the native passthrough. Two quirks: the Anthropic SDK wants
# an api_key, so it gets a placeholder and the real credential goes in
# Authorization; and the Unity Catalog connection holding your Anthropic key is
# named in Databricks-Model-Provider-Service.
from anthropic import Anthropic

client = sdk.wrap(Anthropic(
    api_key="unused",
    base_url=f"{DBX_HOST}/ai-gateway/anthropic",
    default_headers={
        "Authorization": f"Bearer {DBX_TOKEN}",
        "Databricks-Model-Provider-Service": SVC_ANTHROPIC,
        "Databricks-Ai-Gateway-Request-Tags": TAGS,
    },
))
resp = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=400,
    messages=[{"role": "user", "content": PROMPT}],
)
print("resolved model:", resp.model, "| usage:", resp.usage)
print(resp.content[0].text)


In [ ]:
# OpenAI BYOK, via the native OpenAI surface. Same client class as the hosted cell
# above — only base_url differs, and that difference decides which price table
# applies. This path is priced from OpenRouter and matched Databricks' own metered
# spend to the digit on every real call tested.
from openai import OpenAI

client = sdk.wrap(OpenAI(
    api_key=DBX_TOKEN,
    base_url=f"{DBX_HOST}/ai-gateway/openai/v1",
    default_headers={
        "Databricks-Model-Provider-Service": SVC_OPENAI,
        "Databricks-Ai-Gateway-Request-Tags": TAGS,
    },
))
resp = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": PROMPT}],
    max_tokens=400,
)
print("resolved model:", resp.model, "| usage:", resp.usage)
print(resp.choices[0].message.content)

assert sdk.flush(timeout=30.0), "queue did not flush in time"
print("\nflushed — check Lago for llm_cost / token events")
